# What is a Subquery
In SQL, a subquery is a query within another query. It is a SELECT statement that is
nested inside another SELECT, INSERT, UPDATE, or DELETE statement. The
subquery is executed first, and its result is then used as a parameter or condition
for the outer query.
- Note - The topic is slightly difficult and needs a lot of practice

In [ ]:
CREATE TABLE movies (
    name VARCHAR(255),
    rating VARCHAR(10),
    genre VARCHAR(100),
    year INTEGER,
    released VARCHAR(100),
    score DECIMAL(3,1),
    votes DECIMAL(12,1),
    director VARCHAR(255),
    writer VARCHAR(255),
    star VARCHAR(255),
    country VARCHAR(100),
    budget BIGINT,
    gross DECIMAL(15,1),
    company VARCHAR(255),
    runtime DECIMAL(5,1)
);

In [ ]:
-- example 
SELECT * FROM public.movies
WHERE score = (SELECT MAX(score) FROM public.movies)

-- outer query (it runs after inner query)
-- inner query (it runs first) (SELECT MAX(score) FROM public.movies)

# Types of Subqueries

### 1. Based on the result it returns (Returned Data)
* **Scalar subquery**: Returns a single value.
* **Row subquery**: Returns a single row of data.
* **Table subquery**: Returns a multi-row, multi-column table.

### 2. Based on working
* **Independent**: Subquery executes on its own.
* **Correlated**: Subquery depends on the outer query.


# Where Can Subqueries Be Used?

### 1. SELECT
* **`SELECT`** – to calculate a specific column value.
* **`FROM`** – to create a temporary table.
* **`WHERE`** – to filter individual rows.
* **`HAVING`** – to filter grouped data.

### 2. INSERT
* To copy and insert data from another table.

### 3. UPDATE
* To calculate new values or filter rows during an update.

### 4. DELETE
* To filter which specific rows should be removed.


## Independent Subquery - Scalar Subquery
1. Find the movie with highest profit(vs order by)
2. Find how many movies have a rating > the avg of all the movie ratings(Find the count of above average movies)
3. Find the highest rated movie of 2000
4. Find the highest rated movie among all movies whose number of votes are >
the dataset avg votes

In [ ]:
-- 1 
SELECT * FROM movies
WHERE (gross - budget) = (SELECT MAX(gross - budget) FROM movies);

-- and othe way without subquery 
SELECT * FROM movies
ORDER BY (gross - budget) DESC LIMIT 1


In [ ]:
-- 2
SELECT COUNT(*) FROM movies
WHERE score > (SELECT AVG(score) FROM movies)


In [ ]:
-- 3
SELECT * FROM movies
WHERE year = 2000 AND score = (SELECT MAX(score) FROM movies
WHERE year = 2000)


In [ ]:
-- 4 
SELECT * FROM movies 
WHERE score = (SELECT MAX(score) FROM movies 
				WHERE votes > (SELECT AVG(votes) 
								FROM movies))


## Independent Subquery - Row Subquery(One Col Multi Rows)
1. Find all users who never ordered
2. Find all the movies made by top 3 directors(in terms of total gross income)
3. Find all movies of all those actors whose filmography's avg rating > 8.5(take 25000 votes as cutoff)

In [ ]:
CREATE TABLE users (
    user_id SERIAL PRIMARY KEY,
    name VARCHAR(100) NOT NULL,
    email VARCHAR(255) UNIQUE NOT NULL,
    password VARCHAR(100) NOT NULL
);

In [ ]:
CREATE TABLE restaurants (
    r_id SERIAL PRIMARY KEY,
    r_name VARCHAR(100) NOT NULL,
    cuisine VARCHAR(100) NOT NULL
);

In [ ]:
CREATE TABLE orders (
    order_id INT PRIMARY KEY,
    user_id INT,
    r_id INT,
    amount NUMERIC,
    date DATE,
    partner_id INT,
    delivery_time INT,
    delivery_rating INT,
    restaurant_rating INT
);

In [ ]:
CREATE TABLE order_details (
    id INT PRIMARY KEY,
    order_id INT,
    f_id INT
);


In [ ]:
CREATE TABLE menu (
    menu_id INT PRIMARY KEY,
    r_id INT,
    f_id INT,
    price NUMERIC
);

In [ ]:
CREATE TABLE food (
    f_id INT PRIMARY KEY,
    f_name VARCHAR(100),
    type VARCHAR(50)
);

In [ ]:
CREATE TABLE delivery_partners (
    partner_id INT PRIMARY KEY,
    partner_name VARCHAR(100)
);

In [ ]:
-- 1 
SELECT * FROM public.users
WHERE user_id NOT IN (SELECT DISTINCT(user_id) 
							FROM public.orders)


In [ ]:
-- 2
SELECT * FROM movies
WHERE director IN (
    SELECT director 
    FROM public.movies
    WHERE gross IS NOT NULL 
      AND director IS NOT NULL
    GROUP BY director
    ORDER BY SUM(gross) DESC 
    LIMIT 3
)
ORDER BY gross DESC;


In [ ]:
-- 3
SELECT * FROM movies
WHERE star IN (SELECT star FROM movies
				WHERE votes > 25000
				GROUP BY star
				HAVING AVG(score) > 8.5)

### Independent Subquery - Table Subquery (Multi Col Multi Row)
1. Find the most profitable movie of each year
2. Find the highest rated movie of each genre votes cutoff of 25000
3. Find the highest grossing movies of top 5 actor/director combo in terms of
total gross income

In [ ]:
-- 1 
SELECT * FROM movies
WHERE (year, gross - budget) IN (SELECT year,
								MAX(gross - budget)
								FROM public.movies
									GROUP BY year)


In [ ]:
-- 2
SELECT * FROM public.movies
WHERE (genre, score) IN (SELECT genre, MAX(score)
						FROM public.movies
						WHERE votes > 25000
						GROUP BY genre)
AND votes > 25000


In [ ]:
-- 3
SELECT * FROM movies
WHERE (star, director, gross) IN (SELECT star,director, MAX(gross) FROM public.movies
WHERE gross IS NOT NULL 
GROUP BY star, director
ORDER BY SUM(gross) DESC LIMIT 5)

## Correlated Subquery
1. Find all the movies that have a rating higher than the average
rating of movies in the same genre.
2. Find the favorite food of each customer.

In [ ]:
-- 1
SELECT * FROM movies AS m1
WHERE score > (SELECT AVG(score) FROM movies AS m2
				WHERE m2.genre = m1.genre)



In [ ]:
-- 2
SELECT * FROM public.users
WHERE user_id NOT IN (SELECT DISTINCT(user_id)
						FROM orders);

WITH fav_food AS (
SELECT t2.user_id, name, t4.f_name, COUNT(*) AS frequency FROM users AS t1
JOIN orders AS t2 ON t1.user_id = t2.user_id
JOIN order_details AS t3 ON t2.order_id = t3.order_id
JOIN food AS t4 ON t3.f_id = t4.f_id
GROUP BY t2.user_id, name, t4.f_name
)

SELECT * FROM fav_food AS f1
WHERE frequency = ( SELECT MAX(frequency)
					FROM fav_food AS f2
					WHERE f2.user_id = f1.user_id)


## Usage with SELECT
1. Get the percentage of votes for each movie compared to the total number of
votes.
2. Display all movie names ,genre, score and avg(score) of genre


In [ ]:
-- 1 
SELECT * FROM movies AS m1
WHERE score > (SELECT AVG(score) FROM movies AS m2 WHERE m2.genre = m1.genre);

SELECT name, (votes/(SELECT SUM(votes) FROM movies)) * 100 FROM movies

In [ ]:
-- 2
SELECT name, genre, score,
(SELECT AVG(score) FROM movies AS m2 
WHERE m2.genre = m1.genre)
FROM movies AS m1

## Usage with FROM
1. Display average rating of all the restaurants

In [ ]:
SELECT r_name, avg_rating
FROM (SELECT r_id, AVG(restaurant_rating) AS avg_rating
FROM orders
GROUP By r_id) t1 JOIN restaurants t2
ON t1.r_id = t2.r_id


## Usage with HAVING
1. Find genres having avg score > avg score of all the movies

In [ ]:
SELECT genre, AVG(score)
FROM movies
GROUP BY genre
HAVING AVG(score) > (SELECT AVG(score) FROM movies)

## Subquery In INSERT
1. Populate a already created loyal_customers table with records of only
those customers who have ordered food more than 3 times.

## Subquery in UPDATE
1. Populate the money col of loyal_cutomer table using the orders table.
Provide a 10% app money to all customers based on their order value.

## Subquery in DELETE
1. Delete all the customers record who have never ordered.